## Module 4-4 FinBERT Sentiment

### A two-line glossary

- **Pretraining**: FinBERT was first trained (like Google's BERT) to predict masked-out words in a huge pile of unlabeled text -- except FinBERT's pile is 4.9 billion words of 10-Ks, 10-Qs, analyst reports, and earnings call transcripts, instead of Wikipedia. This is what teaches it finance vocabulary ("liquidity," "amortization," "headwind") and typical financial phrasing.
- **Fine-tuning**: starting from that pretrained model, a small labeled dataset (e.g. sentences labeled positive/neutral/negative) is used to adapt it to a specific task. We use an *already fine-tuned* version of FinBERT below; in 4-4o we fine-tune one ourselves.

### 1. Load FinBERT and classify a few sentences

`yiyanghkust/finbert-tone` is FinBERT fine-tuned for sentiment classification, released on the Hugging Face Hub by the paper's authors. The first time you run this cell it downloads the model (~440MB); after that it is cached locally.

Note: we load `BertTokenizer`/`BertForSequenceClassification` directly (rather than the more generic `transformers.pipeline` / `AutoModel`) because this model's `config.json` predates a metadata field that newer versions of `transformers` expect for auto-detection -- loading the concrete BERT classes sidesteps that.

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import pandas as pd

finbert = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone', num_labels=3)
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')
finbert.eval()

LABELS = {0: 'neutral', 1: 'positive', 2: 'negative'}

In [ ]:
def classify_sentences(sentences: list[str], batch_size: int = 32) -> list[str]:
    """Return FinBERT's predicted sentiment label for each sentence."""
    labels = []
    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i + batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=128)
            outputs = finbert(**inputs)[0]
            preds = torch.argmax(outputs, dim=1).tolist()
            labels.extend(LABELS[p] for p in preds)
    return labels

In [ ]:
demo_sentences = [
    'there is a shortage of capital, and we need extra financing',
    'growth is strong and we have plenty of liquidity',
    'there are doubts about our finances',
    'profits are flat',
]

for sentence, label in zip(demo_sentences, classify_sentences(demo_sentences)):
    print(f'{label:>10} -- {sentence}')

### 2. FinBERT vs. the LM dictionary, sentence by sentence

We reuse the R&D-flagged sentences from 4-2/4-3 (`data/rnd_disclosures.csv`) and score each sentence two ways: with FinBERT, and with a simple Loughran-McDonald rule (negative if it contains any LM negative word; positive if it contains an LM positive word and no negative word; neutral otherwise -- the same rule used for `Tone` in the paper and in `data/rnd_disclosures.csv`'s own dictionary counts).

In [ ]:
sheet_id = '1y2LVPvRqdggmIhSnHQcEZA5lYbe3vS5w'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv'
LM_Dictionary = pd.read_csv(url)

LM_Neg_Words = set(LM_Dictionary.loc[LM_Dictionary['Negative'] != 0, 'Word'])
LM_Pos_Words = set(LM_Dictionary.loc[LM_Dictionary['Positive'] != 0, 'Word'])

def lm_label(sentence: str) -> str:
    words = {w.upper() for w in sentence.split()}
    if words & LM_Neg_Words:
        return 'negative'
    if words & LM_Pos_Words:
        return 'positive'
    return 'neutral'

In [ ]:
rnd_df = pd.read_csv('data/rnd_disclosures.csv')

records = []
for _, row in rnd_df.iterrows():
    for sentence in str(row['RnD_Text']).split('\n'):
        sentence = sentence.strip()
        if len(sentence) > 0:
            records.append({'CIK': row['CIK'], 'sentence': sentence})

sentences_df = pd.DataFrame(records)
sentences_df['finbert_label'] = classify_sentences(sentences_df['sentence'].tolist())
sentences_df['lm_label'] = sentences_df['sentence'].apply(lm_label)

print(sentences_df.shape)
sentences_df.head()

In [ ]:
# How often do the two methods agree?
pd.crosstab(sentences_df['lm_label'], sentences_df['finbert_label'], margins=True)

In [ ]:
# Sentences where FinBERT and the LM dictionary disagree most starkly
disagreements = sentences_df[
    ((sentences_df['lm_label'] == 'positive') & (sentences_df['finbert_label'] == 'negative')) |
    ((sentences_df['lm_label'] == 'negative') & (sentences_df['finbert_label'] == 'positive'))
]
for _, row in disagreements.head(5).iterrows():
    print(f"LM: {row['lm_label']:<9} FinBERT: {row['finbert_label']:<9} {row['sentence']}")

Look closely at a few disagreements: the LM dictionary reacts to the *presence* of a trigger word regardless of how it's used (negation, a word describing a risk being *managed*, boilerplate legal language), while FinBERT reads the sentence as a whole. This is the same mechanism behind the paper's word-order-randomization test (Table 3): scramble the words and FinBERT's accuracy collapses by 11 points, while the dictionary is completely unaffected -- because the dictionary was never using word order in the first place.

### 3. Document-level tone

The paper defines a document's *Tone* as the share of positive sentences minus the share of negative sentences (Section 5). We compute the same measure for each filing, using FinBERT's sentence-level labels.

In [ ]:
def tone_score(labels: list[str]) -> float:
    n = len(labels)
    pct_pos = labels.count('positive') / n
    pct_neg = labels.count('negative') / n
    return pct_pos - pct_neg

filing_tone = (
    sentences_df
    .groupby('CIK')['finbert_label']
    .apply(list)
    .apply(tone_score)
    .rename('FinBERT_Tone')
)
filing_tone

### 4. Applying it to a real earnings call

`data/conference_call_transcript/` has two real Tesla earnings-call transcripts (Q4 2025 and Q1 2026), scraped from Seeking Alpha. Real transcripts are messy Markdown (YAML headers, bold speaker names, italic titles, stray formatting), so we write a small helper to pull out just the spoken text before sentence-tokenizing it.

For simplicity, we score *every* sentence in the call (executives and analysts alike). The paper's own `Tone` measure covers only managers' remarks (Section 5) -- as a follow-up exercise, try filtering to sentences spoken by the participants listed at the top of each transcript file.

In [ ]:
import re
from pathlib import Path
from nltk.tokenize import sent_tokenize

def load_transcript_sentences(path) -> list[str]:
    text = Path(path).read_text(encoding='utf-8')

    if text.startswith('---'):  # drop YAML frontmatter, if present
        parts = text.split('---', 2)
        if len(parts) >= 3:
            text = parts[2]

    match = re.search(r'(?:^#\s*Presentation|^\*\*Presentation\*\*)', text, flags=re.MULTILINE)
    if match:  # drop the title/participant-list boilerplate before the call itself starts
        text = text[match.start():]

    text = re.sub(r'^#.*$', '', text, flags=re.MULTILINE)            # section headings
    text = re.sub(r'^\*\*.+?\*\*\s*$', '', text, flags=re.MULTILINE)  # **Speaker Name** lines
    text = re.sub(r'^\*.+?\*\s*$', '', text, flags=re.MULTILINE)      # *Title* lines
    text = text.replace('**', '').replace('*', '')                   # stray emphasis markers
    text = re.sub(r'\[Operator Instructions\]', '', text)

    return [s.strip() for s in sent_tokenize(text) if len(s.strip()) > 15]

In [ ]:
transcript_files = {
    '2025Q4': '../data/conference_call_transcript/TSLA_2025Q4.md',
    '2026Q1': '../data/conference_call_transcript/TSLA_2026Q1.md',
}

call_tone = {}
for quarter, path in transcript_files.items():
    sentences = load_transcript_sentences(path)
    labels = classify_sentences(sentences)
    call_tone[quarter] = {'n_sentences': len(sentences), 'FinBERT_Tone': tone_score(labels)}
    print(f'{quarter}: {len(sentences)} sentences, Tone = {tone_score(labels):.3f}')

pd.DataFrame(call_tone).T

This is a miniature version of the paper's Section 5 analysis, which runs the same calculation across 28,873 earnings calls and shows that `Tone_FinBERT` explains next-day market reactions better than tone measured by the LM dictionary or any of the classic ML models (Table 7).

### Next: fine-tuning FinBERT yourself

Everything above used an *already fine-tuned* FinBERT. In `4-4o_FinBERT_Finetuning.ipynb` (optional) we fine-tune the pretrained FinBERT ourselves on a small labeled dataset, and see how little training data it actually needs to perform well -- directly replicating the paper's small-sample-size finding (Table 2).